# Rerank Latency Test

Learn more about Cohere's Reranking API [here](https://docs.cohere.com/docs/rerank-overview).

We benchmark reranking latency for Cohere (rerank-v4.0-pro) using text transcriptions from the IRPAPERS dataset. For each trial, we randomly sample k documents and a query from the dataset, then measure wall-clock time for a single rerank call returning the top 10 results. We sweep k across [10, 20, 50, 100, 200, 500] with 5 random samples per k value to control for content-dependent variation and potential server-side caching. All measurements use time.perf_counter() for monotonic precision. 

### TLDR: The Results

| k | Mean (s) | Std (s) | Min (s) | Max (s) |
|---:|---:|---:|---:|---:|
| 10 | 0.403 | 0.045 | 0.366 | 0.481 |
| 20 | 0.605 | 0.028 | 0.569 | 0.642 |
| 50 | 0.840 | 0.080 | 0.713 | 0.908 |
| 100 | 1.162 | 0.266 | 1.013 | 1.636 |
| 200 | 1.474 | 0.447 | 1.140 | 2.153 |
| 500 | 2.296 | 0.070 | 2.222 | 2.375 |

In [ ]:
import os
import time
import random
import statistics
 
import cohere
import numpy as np
from datasets import load_dataset

os.environ["COHERE_API_KEY"] = "YOUR-COHERE-API-KEY"

In [28]:
MODEL = "rerank-v4.0-pro"
TOP_N = 10
DOC_COUNTS = [10, 20, 50, 100, 200, 500]  # ablation over k
SAMPLES_PER_K = 5                          # random draws per k
SEED = 42
 
co = cohere.Client(os.environ["COHERE_API_KEY"])

In [29]:
print("Loading IRPAPERS dataset...")
docs_ds = load_dataset("weaviate/IRPAPERS", "docs", split="train")
queries_ds = load_dataset("weaviate/IRPAPERS", "queries", split="train")
 
all_docs = [doc["transcription"] for doc in docs_ds]
all_queries = [q["question"] for q in queries_ds]
 
print(f"  {len(all_docs)} docs, {len(all_queries)} queries loaded\n")

Loading IRPAPERS dataset...
  3230 docs, 180 queries loaded



In [30]:
def sample_docs(docs: list[str], k: int, rng: random.Random) -> list[str]:
    """Draw k docs without replacement. Caps at len(docs)."""
    k = min(k, len(docs))
    return rng.sample(docs, k)
 
 
def sample_query(queries: list[str], rng: random.Random) -> str:
    return rng.choice(queries)

In [31]:
def bench_rerank(query: str, documents: list[str]) -> float:
    """Time a single rerank call. Returns wall-clock seconds."""
    t0 = time.perf_counter()
    _ = co.rerank(model=MODEL, query=query, documents=documents, top_n=TOP_N)
    return time.perf_counter() - t0

In [35]:
def run_ablation(
    all_docs: list[str],
    all_queries: list[str],
    doc_counts: list[int],
    samples_per_k: int,
    seed: int,
) -> list[dict]:
    """Run the full ablation. Returns list of result dicts."""
    rng = random.Random(seed)
    results = []
 
    for k in doc_counts:
        times = []
        for trial in range(samples_per_k):
            docs_sample = sample_docs(all_docs, k, rng)
            query = sample_query(all_queries, rng)
            elapsed = bench_rerank(query, docs_sample)
            times.append(elapsed)
            print(f"  k={k:>4d}  trial={trial+1}/{samples_per_k}  {elapsed:.3f}s")
 
        results.append({
            "k": k,
            "mean_s": statistics.mean(times),
            "std_s": statistics.stdev(times) if len(times) > 1 else 0.0,
            "min_s": min(times),
            "max_s": max(times),
            "times": times,
        })
 
    return results

In [36]:
print(f"Ablating over doc counts: {DOC_COUNTS}")
print(f"  {SAMPLES_PER_K} random samples per k, seed={SEED}\n")
 
results = run_ablation(all_docs, all_queries, DOC_COUNTS, SAMPLES_PER_K, SEED)

Ablating over doc counts: [10, 20, 50, 100, 200, 500]
  5 random samples per k, seed=42

  k=  10  trial=1/5  0.481s
  k=  10  trial=2/5  0.380s
  k=  10  trial=3/5  0.402s
  k=  10  trial=4/5  0.366s
  k=  10  trial=5/5  0.385s
  k=  20  trial=1/5  0.569s
  k=  20  trial=2/5  0.642s
  k=  20  trial=3/5  0.605s
  k=  20  trial=4/5  0.619s
  k=  20  trial=5/5  0.592s
  k=  50  trial=1/5  0.857s
  k=  50  trial=2/5  0.908s
  k=  50  trial=3/5  0.713s
  k=  50  trial=4/5  0.819s
  k=  50  trial=5/5  0.902s
  k= 100  trial=1/5  1.045s
  k= 100  trial=2/5  1.636s
  k= 100  trial=3/5  1.013s
  k= 100  trial=4/5  1.057s
  k= 100  trial=5/5  1.058s
  k= 200  trial=1/5  1.214s
  k= 200  trial=2/5  1.152s
  k= 200  trial=3/5  2.153s
  k= 200  trial=4/5  1.711s
  k= 200  trial=5/5  1.140s
  k= 500  trial=1/5  2.368s
  k= 500  trial=2/5  2.375s
  k= 500  trial=3/5  2.263s
  k= 500  trial=4/5  2.253s
  k= 500  trial=5/5  2.222s


In [37]:
print("\n" + "=" * 60)
print(f"{'k':>6s}  {'mean':>8s}  {'std':>8s}  {'min':>8s}  {'max':>8s}")
print("-" * 60)
for r in results:
    print(f"{r['k']:>6d}  {r['mean_s']:>7.3f}s  {r['std_s']:>7.3f}s  {r['min_s']:>7.3f}s  {r['max_s']:>7.3f}s")
print("=" * 60)


     k      mean       std       min       max
------------------------------------------------------------
    10    0.403s    0.045s    0.366s    0.481s
    20    0.605s    0.028s    0.569s    0.642s
    50    0.840s    0.080s    0.713s    0.908s
   100    1.162s    0.266s    1.013s    1.636s
   200    1.474s    0.447s    1.140s    2.153s
   500    2.296s    0.070s    2.222s    2.375s
